# **Neural Networks Day 4 - PyTorch Version**

Welcome to the Neural Network practice with **PyTorch**!

This notebook covers Convolutional Neural Networks (CNNs) using PyTorch. You'll learn how to build and train CNNs, including VGG and ResNet architectures.

## **Module Imports**

In PyTorch, we import the core `torch` library along with:
- `torch.nn`: Neural network layers and loss functions
- `torch.nn.functional`: Functional API for operations like activations
- `torch.optim`: Optimization algorithms (SGD, Adam, etc.)
- `torchvision`: Datasets and transformations for computer vision
- `torch.utils.data.DataLoader`: Efficient data loading and batching

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# PyTorch core imports
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

# torchvision for datasets and transforms
import torchvision
import torchvision.transforms as transforms

# Check if GPU is available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

---

## **Understanding Conv2d in PyTorch**

Before diving into exercises, let's understand how convolutional layers work in PyTorch:

### `nn.Conv2d(in_channels, out_channels, kernel_size, stride, padding)`

| Parameter | Description | Example for MNIST |
|-----------|-------------|-------------------|
| `in_channels` | Number of input channels | 1 (grayscale) or 3 (RGB) |
| `out_channels` | Number of filters/kernels to apply | 32, 64, etc. |
| `kernel_size` | Size of the convolutional kernel | 3, 5, 7, or (3,3), (5,5) |
| `stride` | Step size when sliding the kernel | 1, 2 |
| `padding` | Pixels to add around input ('same' or number) | 0 ('valid'), 1, 2 |

### Key PyTorch Conv2d Notes:
- Specify `in_channels` explicitly (PyTorch doesn't infer it)
- **Padding**: Use integers (0='valid', >0 adds that many pixels per side) or 'same' in newer versions
- **Activation**: Applied separately using `F.relu()` or `nn.ReLU()` layer

### Output Size Formula:
```
Output_size = floor((Input_size + 2*padding - kernel_size) / stride) + 1
```

---

## **Understanding MaxPool2d**

### `nn.MaxPool2d(kernel_size, stride)`

Max pooling reduces spatial dimensions by taking the maximum value in each pooling window:

```python
nn.MaxPool2d(kernel_size=2, stride=2)  # Reduces HxW by half
```

This reduces spatial dimensions by taking the maximum value in each 2×2 window.

---

## **Building CNNs in PyTorch**

In PyTorch, models are typically defined as classes inheriting from `nn.Module`:

```python
class MyModel(nn.Module):
    def __init__(self):
        super(MyModel, self).__init__()
        self.conv1 = nn.Conv2d(1, 32, kernel_size=3, padding=1)
        self.pool = nn.MaxPool2d(2, 2)
        self.fc1 = nn.Linear(32 * 14 * 14, 10)
    
    def forward(self, x):
        x = self.pool(F.relu(self.conv1(x)))
        x = x.view(x.size(0), -1)  # Flatten
        x = self.fc1(x)
        return x
```

### Key Points:
- `__init__`: Define all layers
- `forward()`: Define the forward pass computation
- `x.view()`: Reshape tensors (flatten for fully connected layers)

---

## **Training Loop in PyTorch**

PyTorch requires explicit training loops (unlike high-level framework APIs):

```python
# Training loop
for epoch in range(num_epochs):
    model.train()
    for batch_idx, (data, target) in enumerate(train_loader):
        data, target = data.to(device), target.to(device)
        
        optimizer.zero_grad()      # Clear gradients
        output = model(data)       # Forward pass
        loss = criterion(output, target)  # Compute loss
        loss.backward()            # Backward pass
        optimizer.step()           # Update weights
```

---

# **Question 2: Dense vs Conv2D Parameter Comparison**

**YOUR TASK**: 

Load the MNIST dataset using `torchvision.datasets.MNIST` and initialize two models.

### Model 1 (Dense):
- Input: MNIST images (28x28)
- Flatten
- Linear layer: 10 units (equivalent to dense with tanh)
- Linear layer: 10 units (output with softmax)

### Model 2 (CNN):
- Input: MNIST images (1x28x28)
- Conv2d: 10 filters, 5x5 kernel, stride 1, padding 0
- Flatten
- Linear layer: 10 units (output)

**Compare the number of parameters** from the two models.

Then duplicate the hidden layer in both models and observe how parameter counts increase.

### Hints:
- Use `sum(p.numel() for p in model.parameters())` to count parameters
- Remember: Conv2d layer output size = (28-5+1) = 24, so 10 filters → 10×24×24 = 5760 features before flatten
- For second Conv2d layer, calculate input channels based on first layer's output

In [ ]:
# ============================================================
# EXERCISE FOR STUDENTS: Implement Dense vs Conv2D comparison
# ============================================================

# Data parameters
num_classes = 10
input_shape = (1, 28, 28)  # PyTorch format: (channels, height, width)

# TODO: Load MNIST dataset using torchvision.datasets.MNIST
# Hint: Use transforms.ToTensor() to normalize to [0,1]


# TODO: Create DataLoaders for batching


# TODO: Define Dense Model
class DenseModel(nn.Module):
    pass  # Your implementation here


# TODO: Define CNN Model
class CNNModel(nn.Module):
    pass  # Your implementation here


# TODO: Instantiate models and compare parameter counts


# TODO: Define models with DUPLICATE hidden layers and compare again


---

## **Understanding VGG Blocks in PyTorch**

A VGG block typically consists of:
1. Multiple Conv2d layers with small kernels (3×3)
2. ReLU activation after each
3. MaxPool2d at the end for downsampling

```python
class VGGBlock(nn.Module):
    def __init__(self, in_channels, out_channels):
        super(VGGBlock, self).__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2)
        )
    
    def forward(self, x):
        return self.block(x)
```

---

## **Data Augmentation with torchvision.transforms**

PyTorch provides powerful data augmentation through `torchvision.transforms`:

```python
# For training: apply random augmentations
train_transform = transforms.Compose([
    transforms.RandomRotation(10),
    transforms.RandomAffine(degrees=0, translate=(0.1, 0.1)),
    transforms.ToTensor(),
])

# For testing: only normalize
test_transform = transforms.Compose([
    transforms.ToTensor(),
])
```

For the custom padding augmentation required in this exercise, we'll implement a custom function.

---

# **Question 5: VGG Block with Data Augmentation**

**YOUR TASK**: 

Implement a VGG block using PyTorch layers:

### VGG Block Structure:
- Conv2d: 8 filters, kernel_size=5, stride=1, padding='same', ReLU
- Conv2d: 8 filters, kernel_size=3, stride=1, padding='same', ReLU
- MaxPool2d: size 2×2, stride 2

### Full Model:
- Input (1×56×56 after augmentation)
- VGG block(s)
- Flatten
- Linear output layer (10 classes)

### Data Augmentation:
Use the `loc_noise` function to pad images to 56×56:
- Training: pad to bottom-right (non-random)
- Testing: pad randomly

Train the model and evaluate on the augmented test set.

### Questions to Answer:
- Does one VGG block achieve >95% accuracy?
- How does adding more blocks affect performance?
- How does the model handle the augmented data?

In [ ]:
# ============================================================
# EXERCISE FOR STUDENTS: VGG Block Implementation
# ============================================================

# Function to pad input examples
def loc_noise(data, rnd=True):
    """
    Pads MNIST images (28x28) to 56x56
    
    Args:
        data: numpy array of shape (N, 28, 28)
        rnd: if True, place image randomly; if False, place at bottom-right
    
    Returns:
        numpy array of shape (N, 56, 56)
    """
    shifted = np.zeros((data.shape[0], 56, 56))
    for ex in range(len(data)):
        if rnd:
            ud = np.random.randint(0, 28)
            lr = np.random.randint(0, 28)
            padded = np.pad(data[ex], ((ud, 28-ud), (lr, 28-lr)))
        else:
            padded = np.pad(data[ex], ((28, 0), (28, 0)))
        shifted[ex] = padded
    return shifted


# TODO: Load fresh MNIST data


# TODO: Apply loc_noise augmentation
# - Train: non-random (rnd=False)
# - Test: random (rnd=True)


# TODO: Convert to PyTorch tensors and create DataLoaders
# Remember: PyTorch expects (N, C, H, W) format


# TODO: Define VGG Block
class VGGBlock(nn.Module):
    """
    VGG Block with:
    - Conv2d (8 filters, 5x5, stride 1, padding='same') + ReLU
    - Conv2d (8 filters, 3x3, stride 1, padding='same') + ReLU
    - MaxPool2d (2x2, stride 2)
    """
    def __init__(self, in_channels, out_channels=8):
        super(VGGBlock, self).__init__()
        # Your implementation here
        pass
    
    def forward(self, x):
        # Your implementation here
        pass


# TODO: Define full VGG Model
class VGGModel(nn.Module):
    """
    Full model with VGG blocks, Flatten, and output Linear layer
    """
    def __init__(self, num_blocks=1, num_classes=10):
        super(VGGModel, self).__init__()
        # Your implementation here
        pass
    
    def forward(self, x):
        # Your implementation here
        pass


# TODO: Instantiate model, define loss and optimizer


# TODO: Implement training loop


# TODO: Train the model and evaluate on augmented test set


---

## **Understanding ResNet and Skip Connections**

ResNet (Residual Network) introduced skip connections that allow gradients to flow directly through the network, enabling training of very deep networks.

### The Core Idea:
Instead of learning `F(x)`, learn the residual `F(x) = H(x) - x`, so:
```
Output = F(x) + x  (skip connection)
```


### Key Points:
- Skip connections help with vanishing gradient problem
- The `forward()` method defines the flow including the skip connection
- Using `F.relu()` (functional API) vs `nn.ReLU()` (module) - both work!

---

## **Functional API in PyTorch (torch.nn.functional)**

PyTorch offers two ways to apply operations:

### 1. Module-based (stateful):
```python
self.relu = nn.ReLU() 
out = self.relu(x)      # Use in forward()
```

### 2. Functional API (stateless):
```python
out = F.relu(x)  # Direct function call, no parameters
out = F.max_pool2d(x, kernel_size=2)
out = F.softmax(x, dim=1)
```

### When to use which:
- **Module-based**: Layers with parameters (Conv2d, Linear, BatchNorm)
- **Functional**: Operations without parameters (ReLU, max_pool2d, softmax)

ResNet blocks benefit from functional API for the addition operation!

---

# **Question 6: ResNet Block with Functional API**

**YOUR TASK**: 

First, randomly augment the training set using `loc_noise(rnd=True)`.

Then, implement a **ResNet block** using PyTorch's functional API.

### ResNet Block Structure:
- Conv2d: 8 filters, 5×5 kernel, stride 1, padding='same' + ReLU
- Conv2d: 8 filters, 5×5 kernel, stride 1, padding='same' (no activation)
- **Skip path**: Conv2d 8 filters, 1×1 kernel, stride 1, padding='same' + ReLU
- **Add**: output of second conv + skip path
- **ReLU**: final activation

### Full ResNet Model:
- Input: 56×56×1 (augmented)
- ResNet block
- MaxPool2d (2×2, stride 2)
- ResNet block
- MaxPool2d (2×2, stride 2)
- ResNet block
- MaxPool2d (2×2, stride 2)
- Flatten + Linear output

Train for 10 epochs and compare with the 3-block VGG model from Question 5.

### Comparison Questions:
- Which achieves better performance on augmented test data?
- How do skip connections help with training?

In [ ]:
# ============================================================
# EXERCISE FOR STUDENTS: ResNet Block Implementation
# ============================================================

# TODO: Load fresh MNIST data and apply RANDOM augmentation to BOTH train and test


# TODO: Define ResNet Block using functional API for skip connection
class ResNetBlock(nn.Module):
    """
    ResNet Block with skip connection:
    - Conv2d (8 filters, 5x5, stride 1, padding='same') + ReLU
    - Conv2d (8 filters, 5x5, stride 1, padding='same') (no activation)
    - Skip: Conv2d (8 filters, 1x1, stride 1, padding='same') + ReLU
    - Add main + skip, then ReLU
    
    Use torch.nn.functional (F) for the addition and final ReLU!
    """
    def __init__(self, channels=8):
        super(ResNetBlock, self).__init__()
        # Define convolutional layers here
        pass
    
    def forward(self, x):
        # Implement:
        # 1. First conv + relu
        # 2. Second conv (no activation)
        # 3. Skip connection with 1x1 conv + relu
        # 4. Add main path and skip using F (functional API)
        # 5. Final relu using F
        pass


# TODO: Define full ResNet Model with 3 blocks and MaxPool
class ResNetModel(nn.Module):
    """
    Full ResNet model:
    - ResNet block + MaxPool
    - ResNet block + MaxPool
    - ResNet block + MaxPool
    - Flatten + Linear output
    """
    def __init__(self, num_classes=10):
        super(ResNetModel, self).__init__()
        # Your implementation here
        pass
    
    def forward(self, x):
        # Your implementation here
        pass


# TODO: Also define a 3-block VGG model for comparison


# TODO: Train ResNet for 10 epochs


# TODO: Compare ResNet vs VGG performance on augmented test set


---

## **Summary: PyTorch CNN Cheatsheet**

| PyTorch Component | Usage |
|-------------------|-------|
| `nn.Conv2d(in_ch, out_ch, kernel_size, ...)` | 2D convolutional layer |
| `nn.MaxPool2d()` | 2D max pooling layer |
| `x.view(x.size(0), -1)` | Flatten tensor for fully connected layers |
| `nn.Linear(in_features, out_features)` | Fully connected (dense) layer |
| `F.relu()` or `nn.ReLU()` | ReLU activation function |
| `optim.Adam()` + `nn.CrossEntropyLoss()` | Optimizer and loss function setup |
| Custom training loop | Training iteration over batches |
| `model.eval()` + `with torch.no_grad()` | Inference/prediction mode |
| `print(model)` or `torchsummary` | Model architecture summary |

### Training Loop Template:
```python
model.train()
for epoch in range(epochs):
    for data, target in train_loader:
        data, target = data.to(device), target.to(device)
        optimizer.zero_grad()
        output = model(data)
        loss = criterion(output, target)
        loss.backward()
        optimizer.step()
```

### Evaluation Template:
```python
model.eval()
correct = 0
total = 0
with torch.no_grad():
    for data, target in test_loader:
        data, target = data.to(device), target.to(device)
        output = model(data)
        _, predicted = torch.max(output.data, 1)
        total += target.size(0)
        correct += (predicted == target).sum().item()
accuracy = 100 * correct / total
```